***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节： [9.2 基础校准流程](9_2_calibration_workflow.ipynb)
    * 下一节： [9.4 自校准实战](9_4_self_calibration.ipynb)

***


## 9.3 连续谱基础成像：从 dirty image 到 restored image

校准之后，连续谱处理进入成像阶段。成像并不是简单地把可见度“画成图”，而是把不完整 Fourier 采样、权重、脏波束和去卷积共同组织成一个可解释的图像产品。实践中至少要区分四个对象：真实天空模型、dirty image、PSF 或 dirty beam，以及去卷积后用恢复波束重建的 restored image。

Dirty image 是采样可见度的反 Fourier 变换；PSF 是同一采样函数对点源的响应。由于 $uv$ 覆盖不完整，dirty image 等于真实天空与 PSF 的卷积再加噪声。CLEAN 的作用是寻找一组稀疏模型分量，使其与 PSF 卷积后解释 dirty image 中的显著结构；restored image 则把模型分量与理想化恢复波束卷积，再加上残差。


### 9.3.1 成像链中的四个核心对象

下面的二维直接 Fourier 实验保留了成像链的关键结构。它不是工业级成像器，但足以说明为什么 dirty image 中的旁瓣和负瓣不能直接解释为天空结构，也说明 restored image 的改善来自去卷积和恢复波束，而不是凭空增加未采样的信息。


![连续谱基础成像链：天空、dirty image、PSF 与 restored image](figures/practical_continuum_imaging_chain.png)

**图 9.3.1** 合成天空、dirty image、PSF 和 restored image。Dirty image 中的条纹来自采样函数；去卷积压低旁瓣后，restored image 的动态范围明显改善，但残差仍保留未建模结构和噪声。


### 9.3.2 成像参数的物理含义

真实软件中的 `cell`、`imsize`、`weighting`、`robust`、`niter`、`threshold`、`mask` 和 `pbcor` 都对应明确的物理或数值选择。像素大小决定图像采样是否足以描述合成波束；图像尺寸决定视场和离轴源是否被纳入；权重决定分辨率与灵敏度的折中；迭代次数和阈值决定去卷积停止条件；mask 决定 CLEAN 允许在哪里放置模型分量；主波束校正决定图像是否代表真实天空亮度而非表观天空。

这些参数只有在校准残差已被控制后才真正有意义。若图像中最强结构仍由带通错误、相位漂移、RFI 或方向依赖效应造成，单纯调整成像参数只会改变伪影外观，而不能解决根本问题。


### 9.3.3 从基础成像走向自校准

连续谱基础成像的直接产物通常还不是最终科学图像。若目标场信噪比足够高，初始图像可以提取天空模型，并返回校准步骤做自校准。这个循环只有在模型可信时才有意义：一个过浅或过错的 CLEAN 模型会把真实天空误差传给增益求解器。下一节因此转入自校准的实践判断，重点讨论什么时候开始、什么时候停止。
